# Tutorial 9: Capstone - Reproducible Scientific Pipeline

Estimated time: 45-60 minutes

## Prerequisites
Core environment ready; optional backends installed if you want full backend verification.

## Learning aims
- Primary package aim: run a complete reproducible workflow and collect proof artifacts
- Secondary scientific aim: practice evidence-based scientific reporting with provenance and tests

## Success criteria
- you can hand over a concise reproducibility report that another lab member can rerun


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Pre-flight: check prior tutorial artifacts

This capstone builds on earlier tutorials. The cell below checks which artifacts exist so you know what to expect.

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)


In [ ]:
import json
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

checks = {
    "Tutorial specs": root / "tutorials" / "specs",
    "Run registry": root / "tmp" / "run_registry.json",
    "Surrogate registry": root / "tmp" / "surrogate_registry.json",
    "Metamodel registry": root / "tmp" / "meta_registry.json",
    "Metamodel samples registry": root / "tmp" / "metamodel_samples_registry.json",
}

print("Pre-flight artifact check:")
for label, p in checks.items():
    exists = p.exists()
    status = "FOUND" if exists else "MISSING (some steps may be skipped)"
    print(f"  {label}: {status}")

## Step 1: Validate model specs and DOE planning


In [ ]:
run_mm_cli('validate', 'tutorials/specs/model.toy.grid.json')
run_mm_cli('validate', 'tutorials/specs/model.biomodels.quick.json')
run_mm_cli('plan', 'tutorials/specs/model.toy.sobol.json')


## Step 2: Repository quality gate


In [ ]:
# Quality gate scoped to the metamodeling package (src/ + tests/).
# `.` would also lint/test the tcr_signaling submodule, which is out of scope here.
run_tool('ruff', 'format', '--check', 'src', 'tests')
run_tool('ruff', 'check', 'src', 'tests')
run_tool('pytest', '-q', '-m', 'not slow', 'tests')

## Step 3: Optional backend verification


In [ ]:
run_tool('pytest', '-q', 'tests/test_surrogate_backends.py', '-k', 'pymc_gp_backend_fit_sample_and_logprob', check=False)
run_tool('pytest', '-q', 'tests/test_surrogate_backends.py', '-k', 'sbi_npe_backend_fit_sample_and_logprob', check=False)


## Step 4: Build a mini report card (graphic)


In [ ]:
import matplotlib.pyplot as plt

labels = ['Spec validation', 'DOE planning', 'Fast tests', 'PyMC check', 'SBI check']
status = [1, 1, 1, 0.5, 0.5]  # adjust manually based on your run outcomes

plt.figure(figsize=(8, 3))
plt.bar(labels, status, color=['tab:green' if s == 1 else 'tab:orange' for s in status])
plt.ylim(0, 1.1)
plt.ylabel('completion status')
plt.title('Capstone reproducibility checklist')
plt.grid(axis='y', alpha=0.3)
plt.show()


## Final deliverable template
Include:
- spec files used,
- run IDs and artifact IDs,
- one scientific interpretation,
- any skipped checks and reasons.

If another lab member can reproduce your outputs from this note, you passed.
